# Module 06 — Convolutional Neural Networks
## 06-03 LeNet-5 & AlexNet

**Objective:** Implement the two landmark CNN architectures — LeCun's LeNet-5 (1998)
and Krizhevsky's AlexNet (2012) — adapted for CIFAR-10, and trace the key design
innovations that drove 14 years of progress in deep learning.

**Prerequisites:** 06-01 (Convolution from Scratch), 06-02 (Pooling & Receptive Fields).


## Part 0 — Setup & Prerequisites

### Historical context

| Year | Architecture | Top-1 (ImageNet) | Key innovations |
|------|-------------|-----------------|-----------------|
| 1998 | **LeNet-5** | — (MNIST/digits) | Shared weights, AvgPool, Tanh |
| 2012 | **AlexNet** | 63.3 % | ReLU, MaxPool, Dropout, data augmentation |

LeNet-5 proved that learnable feature detectors outperform hand-crafted ones on digit
recognition.  AlexNet did the same for natural images at scale, igniting the modern
deep-learning era.

In this notebook we:
1. Implement **LeNet-5** adapted for CIFAR-10 (3-channel, 32×32).
2. Implement a **scaled AlexNet** suitable for CIFAR-10 on CPU.
3. Compare architectures: parameter count, receptive fields, activation functions.
4. Train both on CIFAR-10 and analyse the accuracy gap and its causes.
5. Visualise learned first-layer filters for both models.


In [ ]:
import sys
import math
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

import random
import torchvision.transforms.functional as TF
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA   : {torch.version.cuda}")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


In [ ]:

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE    = 128
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-3
NUM_CLASSES   = 10
DATA_ROOT     = "data/"

# CIFAR-10 per-channel statistics (computed over the training set)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {device}")
print(f"Classes       : {CIFAR_CLASSES}")


### Dataset: CIFAR-10

CIFAR-10 contains 60,000 colour (RGB) $32 \times 32$ images across 10 object classes.
This is the first time we leave grey-scale — the 3-channel input means every
convolutional layer now processes $C_{\text{in}} \times K^2$ weights per filter
instead of $K^2$.

**Official splits:** 50,000 training + 10,000 test.  We carve 10 % of training as
validation (90/10 split), applying data augmentation **only to training images**.

**Training augmentations used:**
- `RandomCrop(32, padding=4)` — random crop with 4-pixel border reflects
- `RandomHorizontalFlip()` — horizontal mirror (objects look the same flipped)
- Normalise with per-channel mean/std


In [ ]:
# ── Data transforms ───────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Load with different transforms for train vs val
full_train_aug = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=train_transform
)
full_train_val = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=False, transform=val_transform
)
test_set = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=val_transform
)

# 90/10 train/val split with consistent index assignment
generator    = torch.Generator().manual_seed(SEED)
all_indices  = torch.randperm(len(full_train_aug), generator=generator).tolist()
train_size   = int(0.9 * len(full_train_aug))
train_indices = all_indices[:train_size]
val_indices   = all_indices[train_size:]

train_set = Subset(full_train_aug, train_indices)   # augmented
val_set   = Subset(full_train_val, val_indices)     # no augmentation

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train : {len(train_set):,}  |  Val : {len(val_set):,}  |  Test : {len(test_set):,}")
print(f"Image shape : {full_train_aug[0][0].shape}  (C x H x W)")

# ── Sample grid: one image per class ─────────────────────────────────────────
raw_cifar = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False,
                                          download=False,
                                          transform=transforms.ToTensor())
class_imgs_raw: dict[int, np.ndarray] = {}
for img_t, lbl in raw_cifar:
    if lbl not in class_imgs_raw:
        class_imgs_raw[lbl] = img_t.permute(1, 2, 0).numpy()  # HxWxC for imshow
    if len(class_imgs_raw) == NUM_CLASSES:
        break

fig_data, axes_data = plt.subplots(1, NUM_CLASSES, figsize=(15, 2.5))
for cls_idx in range(NUM_CLASSES):
    axes_data[cls_idx].imshow(class_imgs_raw[cls_idx])
    axes_data[cls_idx].set_title(CIFAR_CLASSES[cls_idx], fontsize=8)
    axes_data[cls_idx].axis("off")
plt.suptitle("CIFAR-10 — One Sample per Class", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

# ── EDA: class distribution ───────────────────────────────────────────────────
label_counts = Counter(full_train_aug.targets)
print("\nClass distribution (training set — perfectly balanced):")
for cls_idx in range(NUM_CLASSES):
    bar = "=" * (label_counts[cls_idx] // 250)
    print(f"  {CIFAR_CLASSES[cls_idx]:<12s}: {label_counts[cls_idx]:,}  {bar}")

# ── Per-channel pixel statistics ──────────────────────────────────────────────
raw_imgs_np = np.array([raw_cifar[i][0].numpy() for i in range(1000)])  # (1000,3,32,32)
print("\nPer-channel pixel statistics (1000 test images, raw [0,1]):")
print(f"  {'Channel':<10s}  {'mean':>8s}  {'std':>8s}  {'min':>8s}  {'max':>8s}")
for ch, name in enumerate(["Red", "Green", "Blue"]):
    ch_vals = raw_imgs_np[:, ch, :, :].ravel()
    print(f"  {name:<10s}  {ch_vals.mean():>8.4f}  {ch_vals.std():>8.4f}  "
          f"{ch_vals.min():>8.4f}  {ch_vals.max():>8.4f}")


### 0.3 Data Augmentation Visualisation

Training augmentations — `RandomCrop(32, padding=4)` and `RandomHorizontalFlip()` — are applied **only to training images**, not validation or test images.  Each epoch sees a different random crop/flip, effectively multiplying training diversity.


In [ ]:
# ── Data augmentation: effect visualisation ───────────────────────────────────
# Show the same CIFAR-10 image with and without training augmentations to
# illustrate what RandomCrop + RandomHorizontalFlip look like at training time.


raw_img_pil, raw_lbl = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=False, transform=None
)[42]   # pick a fixed sample for reproducibility
raw_lbl_name = CIFAR_CLASSES[raw_lbl]

# Generate 8 augmented versions
aug_transform_demo = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
])

n_aug   = 8
rng_aug = np.random.default_rng(SEED + 99)
aug_imgs = []
for _ in range(n_aug):
    torch.manual_seed(int(rng_aug.integers(0, 2**31)))
    aug_imgs.append(np.array(aug_transform_demo(raw_img_pil)))

fig_aug, axes_aug = plt.subplots(2, 5, figsize=(14, 5.5))
axes_aug[0][0].imshow(np.array(raw_img_pil))
axes_aug[0][0].set_title(f"Original\n'{raw_lbl_name}'", fontsize=9, fontweight="bold")
axes_aug[0][0].axis("off")

for k, aug_img in enumerate(aug_imgs):
    row_idx = (k + 1) // 5
    col_idx = (k + 1) % 5
    axes_aug[row_idx][col_idx].imshow(aug_img)
    axes_aug[row_idx][col_idx].set_title(f"Aug {k+1}", fontsize=8)
    axes_aug[row_idx][col_idx].axis("off")

plt.suptitle(
    f"Data Augmentation: RandomCrop(32, pad=4) + RandomHorizontalFlip\n"
    f"Sample: '{raw_lbl_name}' — 8 augmented versions shown",
    fontsize=10, fontweight="bold",
)
plt.tight_layout()
plt.show()

# Pixel-level statistics: how much do augmentations change pixel values?
orig_np = np.array(raw_img_pil).astype(float)
diffs   = [np.abs(a.astype(float) - orig_np).mean() for a in aug_imgs]
print(f"Mean absolute pixel difference (augmented vs original, [0,255] scale):")
for k, d in enumerate(diffs):
    print(f"  Aug {k+1}: {d:.2f}")
print(f"  Average delta: {np.mean(diffs):.2f} / 255 = {np.mean(diffs)/255:.4f}")
print("\nNote: RandomCrop shifts pixels by up to 4 pixels; HorizontalFlip mirrors left/right.")
print("These augmentations act as a free regulariser — each epoch sees slightly different images.")


### 1.0 Activation Functions: Tanh (LeNet-5) vs ReLU (AlexNet)

Before building the architectures, let's understand why AlexNet switched from Tanh to ReLU.  Tanh **saturates** for large inputs (gradient → 0), while ReLU maintains a constant gradient of 1 for all positive inputs.


In [ ]:
# ── Activation function comparison: Tanh vs ReLU ────────────────────────────
# LeNet-5 uses Tanh; AlexNet uses ReLU.  Let's compare their saturation
# behaviour and gradient magnitude across a range of input values.

x_range = np.linspace(-4, 4, 400)
x_t     = torch.tensor(x_range, dtype=torch.float32)

tanh_out  = torch.tanh(x_t).numpy()
relu_out  = F.relu(x_t).numpy()
tanh_grad = 1.0 - tanh_out ** 2            # d/dx tanh = 1 - tanh^2
relu_grad = (x_range > 0).astype(float)   # d/dx ReLU = 1 if x>0 else 0

fig_act, axes_act = plt.subplots(1, 2, figsize=(12, 4))

axes_act[0].plot(x_range, tanh_out,  label="Tanh", lw=2, color="#1f77b4")
axes_act[0].plot(x_range, relu_out,  label="ReLU", lw=2, color="#d62728", ls="--")
axes_act[0].axhline(0, color="k", lw=0.7, ls=":")
axes_act[0].axvline(0, color="k", lw=0.7, ls=":")
axes_act[0].set_xlabel("Input", fontsize=11)
axes_act[0].set_ylabel("Output", fontsize=11)
axes_act[0].set_title("Activation Output", fontsize=11, fontweight="bold")
axes_act[0].legend(fontsize=10)
axes_act[0].grid(alpha=0.3)

axes_act[1].plot(x_range, tanh_grad, label="Tanh gradient", lw=2, color="#1f77b4")
axes_act[1].plot(x_range, relu_grad, label="ReLU gradient", lw=2, color="#d62728", ls="--")
axes_act[1].axhline(0, color="k", lw=0.7, ls=":")
axes_act[1].set_xlabel("Input", fontsize=11)
axes_act[1].set_ylabel("Gradient magnitude", fontsize=11)
axes_act[1].set_title("Gradient (d/dx activation)", fontsize=11, fontweight="bold")
axes_act[1].legend(fontsize=10)
axes_act[1].grid(alpha=0.3)

plt.suptitle("Tanh vs ReLU: Output and Gradient Comparison",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Quantify saturation: fraction of the range where gradient < 0.1
tanh_sat_frac = np.mean(tanh_grad < 0.1)
relu_sat_frac = np.mean(relu_grad < 0.1)   # only at exactly x=0
print(f"Fraction of [-4,4] where gradient < 0.1:")
print(f"  Tanh : {tanh_sat_frac:.2%}  (saturates for |x| > ~2)")
print(f"  ReLU : {relu_sat_frac:.2%}  (never saturates for positive inputs)")
print()
print("Insight: Tanh saturates in both tails, killing gradients for large activations.")
print("ReLU maintains constant gradient=1 for all positive inputs — key to training")
print("deeper networks without vanishing gradients.")


## Part 1 — LeNet-5 (LeCun et al., 1998)

LeNet-5 was the first practically successful deep CNN, applied to handwritten digit
recognition.  Its key ideas — shared weights, local receptive fields, and sub-sampling
— remain the foundation of every CNN built today.

### Original LeNet-5 layers (for 32×32 MNIST):

| Layer | Type | Channels | Spatial | Activation |
|-------|------|----------|---------|------------|
| C1 | Conv(k=5, s=1) | 1→6 | 32→28 | Tanh |
| S2 | AvgPool(k=2) | 6→6 | 28→14 | — |
| C3 | Conv(k=5, s=1) | 6→16 | 14→10 | Tanh |
| S4 | AvgPool(k=2) | 16→16 | 10→5 | — |
| C5 | Conv(k=5, s=1) | 16→120 | 5→1 | Tanh |
| F6 | FC | 120→84 | — | Tanh |
| Out | FC | 84→10 | — | Softmax |

**Adaptation for CIFAR-10:** input is 3-channel 32×32 (not 1-channel), so C1 accepts
3 input channels.  All other dimensions follow identically.

**Why Tanh?** ReLU was not widely used until 2010.  Tanh is differentiable everywhere
and centred at zero, which helped gradient flow in the shallow networks of the era.


In [ ]:
class LeNet5(nn.Module):
    '''LeNet-5 adapted for CIFAR-10 (3-channel, 32x32 inputs).

    Faithfully implements the 1998 LeCun architecture:
      C1-S2-C3-S4-C5-F6-Output, with Tanh activations and AvgPool subsampling.
    Input channels changed from 1 to 3 for CIFAR-10 compatibility.

    Attributes:
        features: Sequential conv + pool feature extractor.
        classifier: Sequential fully-connected head.
    '''

    def __init__(self, num_classes: int = 10) -> None:
        '''Build LeNet-5 with the original 1998 architecture.

        Args:
            num_classes: Number of output classes (10 for CIFAR-10).
        '''
        super().__init__()
        # Feature extractor: C1-S2-C3-S4-C5
        self.features = nn.Sequential(
            # C1: Conv 3->6, kernel 5x5 | 32x32 -> 28x28
            nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=0),
            nn.Tanh(),
            # S2: AvgPool 2x2 | 28x28 -> 14x14
            nn.AvgPool2d(kernel_size=2, stride=2),

            # C3: Conv 6->16, kernel 5x5 | 14x14 -> 10x10
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.Tanh(),
            # S4: AvgPool 2x2 | 10x10 -> 5x5
            nn.AvgPool2d(kernel_size=2, stride=2),

            # C5: Conv 16->120, kernel 5x5 | 5x5 -> 1x1
            nn.Conv2d(16, 120, kernel_size=5, stride=1, padding=0),
            nn.Tanh(),
        )
        # Classifier: F6 + Output
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Run forward pass: feature extraction then classification.

        Args:
            x: Input images of shape (N, 3, 32, 32).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.classifier(self.features(x))


# ── Verify dimensions ─────────────────────────────────────────────────────────
lenet = LeNet5(num_classes=NUM_CLASSES)
dummy = torch.zeros(2, 3, 32, 32)
with torch.no_grad():
    out = lenet(dummy)
print(f"LeNet-5  input: {dummy.shape}")
print(f"         output: {out.shape}")

# Layer-by-layer spatial trace
x_trace = torch.zeros(1, 3, 32, 32)
print("\nLeNet-5 spatial trace:")
print(f"  {'Layer':<30s}  {'Output shape':>15s}")
for name, layer in lenet.features.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.AvgPool2d, nn.Tanh)):
        x_trace = layer(x_trace)
        layer_name = f"{type(layer).__name__}"
        if isinstance(layer, nn.Conv2d):
            layer_name += f"({layer.in_channels}->{layer.out_channels}, k={layer.kernel_size[0]})"
        print(f"  {layer_name:<30s}  {str(tuple(x_trace.shape)):>15s}")

# Parameter count
lenet_params = sum(p.numel() for p in lenet.parameters())
print(f"\nLeNet-5 total parameters: {lenet_params:,}")


## Part 2 — AlexNet (Krizhevsky et al., 2012)

AlexNet won the 2012 ImageNet challenge with a top-5 error of 15.3 % — 10 percentage
points below the runner-up.  It introduced several ideas now considered standard:

| Innovation | Details |
|------------|---------|
| **ReLU activations** | Replace Tanh; no saturation, faster training |
| **MaxPool** instead of AvgPool | Stronger translation invariance |
| **Dropout** (p=0.5) | First large-scale use as a regulariser |
| **Data augmentation** | Random crops + horizontal flips at training time |
| **GPU training** | Two GTX 580s in parallel — made 60M parameters feasible |

### Scaled AlexNet for CIFAR-10

The original AlexNet expects 224×224 inputs.  We scale it down for 32×32 CIFAR-10:
- Smaller first kernel (3×3 instead of 11×11).
- Fewer channels per layer.
- Same 5-conv + 3-FC structure preserved.


In [ ]:
class AlexNetCIFAR(nn.Module):
    '''AlexNet scaled for CIFAR-10 (3-channel, 32x32 inputs).

    Preserves the 5-conv + 3-FC structure and key AlexNet innovations
    (ReLU, MaxPool, Dropout) while adapting kernel sizes and channel
    widths for 32x32 images.

    Architecture:
      Conv(3->64,k=3,p=1) + ReLU + MaxPool(2)  | 32->16
      Conv(64->192,k=3,p=1) + ReLU + MaxPool(2) | 16->8
      Conv(192->384,k=3,p=1) + ReLU             | 8->8
      Conv(384->256,k=3,p=1) + ReLU             | 8->8
      Conv(256->256,k=3,p=1) + ReLU + MaxPool(2)| 8->4
      FC(256*4*4->1024) + ReLU + Dropout(0.5)
      FC(1024->512) + ReLU + Dropout(0.5)
      FC(512->10)

    Attributes:
        features: Sequential convolutional feature extractor.
        classifier: Sequential fully-connected classifier head.
    '''

    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.5) -> None:
        '''Build AlexNetCIFAR with ReLU, MaxPool, and Dropout.

        Args:
            num_classes: Number of output classes.
            dropout_rate: Dropout probability in FC layers (0.5 in original).
        '''
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 16x16 -> 8x8
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 8x8 (no pool)
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Block 4: 8x8 (no pool)
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Block 5: 8x8 -> 4x4
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Run forward pass through feature extractor and classifier.

        Args:
            x: Input images of shape (N, 3, 32, 32).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.classifier(self.features(x))


# ── Verify dimensions ─────────────────────────────────────────────────────────
alexnet = AlexNetCIFAR(num_classes=NUM_CLASSES)
dummy   = torch.zeros(2, 3, 32, 32)
with torch.no_grad():
    out = alexnet(dummy)
print(f"AlexNet-CIFAR  input: {dummy.shape}")
print(f"               output: {out.shape}")

x_trace = torch.zeros(1, 3, 32, 32)
print("\nAlexNet-CIFAR spatial trace:")
print(f"  {'Layer':<35s}  {'Output':>12s}")
for name, layer in alexnet.features.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.ReLU)):
        x_trace = layer(x_trace)
        lname = type(layer).__name__
        if isinstance(layer, nn.Conv2d):
            lname += f"({layer.in_channels}->{layer.out_channels},k={layer.kernel_size[0]})"
        elif isinstance(layer, nn.MaxPool2d):
            lname += f"(k={layer.kernel_size},s={layer.stride})"
        print(f"  {lname:<35s}  {str(tuple(x_trace.shape)):>12s}")

alexnet_params = sum(p.numel() for p in alexnet.parameters())
print(f"\nAlexNet-CIFAR total parameters: {alexnet_params:,}")


### Architecture Comparison

Before training, we compare the two models on static properties: parameter count,
FLOPs per image, receptive field, and key design choices.


In [ ]:
# ── Side-by-side architecture comparison ─────────────────────────────────────
def compute_rf(layer_configs: list[tuple[int, int]]) -> int:
    '''Compute the final receptive field for a list of (kernel, stride) layers.

    Args:
        layer_configs: List of (kernel_size, stride) per layer.

    Returns:
        Integer final receptive field size in input pixels.
    '''
    rf, jump = 1, 1
    for k, s in layer_configs:
        rf   = rf + (k - 1) * jump
        jump = jump * s
    return rf


def count_flops(model: nn.Module, input_shape: tuple[int, int, int, int]) -> int:
    '''Approximate forward-pass FLOPs (multiply-adds) for Conv2d and Linear layers.

    Args:
        model: PyTorch nn.Module to profile.
        input_shape: Input shape (N, C, H, W) with N=1.

    Returns:
        Total estimated FLOPs for one image.
    '''
    total = 0
    x     = torch.zeros(input_shape)
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            out = layer(x if x.shape[1] == layer.in_channels
                        else torch.zeros(1, layer.in_channels,
                                         x.shape[2], x.shape[3]))
            _, c_out, h_out, w_out = out.shape
            total += 2 * c_out * layer.in_channels * layer.kernel_size[0]**2 * h_out * w_out
        elif isinstance(layer, nn.Linear):
            total += 2 * layer.in_features * layer.out_features
    return total


# Receptive fields (layers: conv + pool/stride)
rf_lenet  = compute_rf([(5,1),(2,2),(5,1),(2,2),(5,1)])       # C1,S2,C3,S4,C5
rf_alexnet = compute_rf([(3,1),(2,2),(3,1),(2,2),(3,1),(3,1),(3,1),(2,2)])  # 5 conv + 3 pool

flops_lenet   = count_flops(LeNet5(),        (1, 3, 32, 32))
flops_alexnet = count_flops(AlexNetCIFAR(),  (1, 3, 32, 32))

compare_rows = [
    {
        "Attribute":      "Year published",
        "LeNet-5":        "1998",
        "AlexNet-CIFAR":  "2012 (adapted)",
    },
    {
        "Attribute":      "Activation",
        "LeNet-5":        "Tanh",
        "AlexNet-CIFAR":  "ReLU",
    },
    {
        "Attribute":      "Pooling",
        "LeNet-5":        "AvgPool",
        "AlexNet-CIFAR":  "MaxPool",
    },
    {
        "Attribute":      "Dropout",
        "LeNet-5":        "None",
        "AlexNet-CIFAR":  "p=0.5 (FC layers)",
    },
    {
        "Attribute":      "Conv layers",
        "LeNet-5":        "3",
        "AlexNet-CIFAR":  "5",
    },
    {
        "Attribute":      "Total parameters",
        "LeNet-5":        f"{sum(p.numel() for p in LeNet5().parameters()):,}",
        "AlexNet-CIFAR":  f"{alexnet_params:,}",
    },
    {
        "Attribute":      "FLOPs / image",
        "LeNet-5":        f"{flops_lenet:,}",
        "AlexNet-CIFAR":  f"{flops_alexnet:,}",
    },
    {
        "Attribute":      "Receptive field",
        "LeNet-5":        f"{rf_lenet} px",
        "AlexNet-CIFAR":  f"{rf_alexnet} px",
    },
    {
        "Attribute":      "Input channels",
        "LeNet-5":        "3 (adapted)",
        "AlexNet-CIFAR":  "3",
    },
]

cmp_df = pd.DataFrame(compare_rows)
print("Architecture comparison:")
print(cmp_df.to_string(index=False))

# Visual bar: parameter ratio
param_ratio = alexnet_params / sum(p.numel() for p in LeNet5().parameters())
print(f"\nAlexNet-CIFAR is {param_ratio:.0f}x larger than LeNet-5 in parameters.")
print(f"FLOPs ratio: {flops_alexnet / max(flops_lenet, 1):.1f}x more compute per image.")


### 2.1 Receptive Field Growth: Layer by Layer

The receptive field (RF) tells us how many input pixels each output neuron can 'see'. Deeper networks accumulate RF across layers — LeNet-5 achieves full-image coverage with 5×5 kernels, while AlexNet uses smaller 3×3 kernels with more layers.


In [ ]:
# ── Receptive field growth: layer-by-layer comparison ────────────────────────
# Show how the receptive field grows layer-by-layer for LeNet-5 and AlexNet,
# revealing why AlexNet's shallower-but-wider design sees more context.

def rf_per_layer(layer_configs: list[tuple[int, int]]) -> list[dict]:
    '''Compute receptive field and jump at each layer.

    Args:
        layer_configs: List of (kernel_size, stride) per layer.

    Returns:
        List of dicts with keys: layer, kernel, stride, rf, jump.
    '''
    rf, jump = 1, 1
    rows = []
    for i, (k, s) in enumerate(layer_configs):
        rf   = rf + (k - 1) * jump
        jump = jump * s
        rows.append({"Layer": i + 1, "k": k, "s": s, "RF": rf, "Jump": jump})
    return rows


lenet_configs  = [(5, 1), (2, 2), (5, 1), (2, 2), (5, 1)]   # C1,S2,C3,S4,C5
alexnet_configs = [(3, 1), (2, 2), (3, 1), (2, 2),           # B1pool,B2pool
                   (3, 1), (3, 1), (3, 1), (2, 2)]           # B3,B4,B5pool

ln_rows = rf_per_layer(lenet_configs)
ax_rows = rf_per_layer(alexnet_configs)

print("LeNet-5 receptive field growth:")
print(f"  {'Layer':>5s}  {'k':>3s}  {'s':>3s}  {'RF':>5s}  {'Jump':>5s}")
for row in ln_rows:
    print(f"  {row['Layer']:>5d}  {row['k']:>3d}  {row['s']:>3d}  {row['RF']:>5d}  {row['Jump']:>5d}")

print("\nAlexNet-CIFAR receptive field growth:")
print(f"  {'Layer':>5s}  {'k':>3s}  {'s':>3s}  {'RF':>5s}  {'Jump':>5s}")
for row in ax_rows:
    print(f"  {row['Layer']:>5d}  {row['k']:>3d}  {row['s']:>3d}  {row['RF']:>5d}  {row['Jump']:>5d}")

# Plot RF growth side by side
fig_rf, ax_rf = plt.subplots(figsize=(9, 4))
ax_rf.plot([r["Layer"] for r in ln_rows], [r["RF"] for r in ln_rows],
           "o-", lw=2, color="#1f77b4", label="LeNet-5", markersize=7)
ax_rf.plot([r["Layer"] for r in ax_rows], [r["RF"] for r in ax_rows],
           "s--", lw=2, color="#d62728", label="AlexNet-CIFAR", markersize=7)
ax_rf.axhline(32, color="k", ls=":", lw=1, label="Image width (32 px)")
ax_rf.set_xlabel("Cumulative conv/pool layers", fontsize=11)
ax_rf.set_ylabel("Receptive field (input pixels)", fontsize=11)
ax_rf.set_title("Receptive Field Growth per Layer", fontsize=11, fontweight="bold")
ax_rf.legend(fontsize=9)
ax_rf.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal RFs: LeNet-5={ln_rows[-1]['RF']} px,  AlexNet-CIFAR={ax_rows[-1]['RF']} px")
print("LeNet-5's C5 collapses 5x5 feature maps to 1x1 — full-field receptive coverage.")
print("AlexNet's last pool reduces to 4x4 — each output still sees part of the image.")


## Part 3 — Training Both Architectures on CIFAR-10

### Training setup

Both models are trained with:
- **Optimiser:** Adam (lr = 1e-3, default betas)
- **Loss:** Cross-entropy
- **Epochs:** 10
- **Augmentation:** RandomCrop + HorizontalFlip (training only — see Part 0)
- **Best model:** tracked by validation loss, restored after training


In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    '''Train for one epoch and return average loss and accuracy.

    Args:
        model: The CNN model.
        dataloader: Training DataLoader.
        criterion: Loss function.
        optimizer: Optimizer instance.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate the model and return average loss and accuracy.

    Args:
        model: The CNN model.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            correct      += logits.argmax(1).eq(labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total


def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    num_epochs: int,
    learning_rate: float,
    device: torch.device,
    model_name: str,
) -> dict:
    '''Full training loop with best-model tracking and test evaluation.

    Args:
        model: The CNN model to train.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader.
        test_loader: Test DataLoader.
        num_epochs: Number of training epochs.
        learning_rate: Adam learning rate.
        device: Compute device.
        model_name: Display name for progress output.

    Returns:
        Dict with keys: train_losses, val_losses, train_accs, val_accs,
        test_acc, test_loss, total_time, model.
    '''
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_losses: list[float] = []
    val_losses:   list[float] = []
    train_accs:   list[float] = []
    val_accs:     list[float] = []

    best_val_loss = float("inf")
    best_state    = None
    t_total       = 0.0

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0
        t_total += elapsed

        train_losses.append(tl); val_losses.append(vl)
        train_accs.append(ta);   val_accs.append(va)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | "
              f"Val Acc: {va:.2%} | Time: {elapsed:.1f}s")

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\nTest Accuracy: {test_acc:.2%}  (best val loss = {best_val_loss:.4f})")

    return {
        "model":       model,
        "train_losses":train_losses,
        "val_losses":  val_losses,
        "train_accs":  train_accs,
        "val_accs":    val_accs,
        "test_acc":    test_acc,
        "test_loss":   test_loss,
        "total_time":  t_total,
    }


In [ ]:
# ── Train LeNet-5 ─────────────────────────────────────────────────────────────
torch.manual_seed(SEED)
lenet_model = LeNet5(num_classes=NUM_CLASSES).to(device)
lenet_hist  = run_training(
    lenet_model, train_loader, val_loader, test_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=device, model_name="LeNet-5",
)

# ── Train AlexNet-CIFAR ────────────────────────────────────────────────────────
torch.manual_seed(SEED)
alexnet_model = AlexNetCIFAR(num_classes=NUM_CLASSES).to(device)
alexnet_hist  = run_training(
    alexnet_model, train_loader, val_loader, test_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=device, model_name="AlexNet-CIFAR",
)


In [ ]:
# ── Training curves: both models ──────────────────────────────────────────────
colors = {"LeNet-5": "#1f77b4", "AlexNet-CIFAR": "#d62728"}
epochs_x = range(1, NUM_EPOCHS + 1)

fig_tr, axes_tr = plt.subplots(1, 2, figsize=(13, 4))
for name, hist in [("LeNet-5", lenet_hist), ("AlexNet-CIFAR", alexnet_hist)]:
    col = colors[name]
    axes_tr[0].plot(epochs_x, hist["val_losses"], color=col, lw=2,
                    marker="o", markersize=4, label=name)
    axes_tr[1].plot(epochs_x, [a*100 for a in hist["val_accs"]], color=col, lw=2,
                    marker="o", markersize=4, label=name)
    # Train curve (dashed)
    axes_tr[0].plot(epochs_x, hist["train_losses"], color=col, lw=1.2,
                    ls="--", alpha=0.5)
    axes_tr[1].plot(epochs_x, [a*100 for a in hist["train_accs"]], color=col, lw=1.2,
                    ls="--", alpha=0.5)

for ax, title, ylabel in [
    (axes_tr[0], "Loss Curves (solid=val, dashed=train)", "Loss"),
    (axes_tr[1], "Accuracy Curves (solid=val, dashed=train)", "Accuracy (%)"),
]:
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("LeNet-5 vs AlexNet-CIFAR — CIFAR-10 Training Curves",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Quick summary
print(f"{'Model':<18s}  {'Val Acc (ep10)':>15s}  {'Test Acc':>10s}  {'Time':>8s}")
print("-" * 58)
for name, hist in [("LeNet-5", lenet_hist), ("AlexNet-CIFAR", alexnet_hist)]:
    print(f"  {name:<16s}  {hist['val_accs'][-1]:>14.2%}  "
          f"{hist['test_acc']:>9.2%}  {hist['total_time']:>7.1f}s")


## Part 4 — Evaluation & Analysis

### 4.1 Test Set Accuracy & Confusion Matrices

We evaluate both models on the held-out CIFAR-10 test set and visualise the confusion
matrices to understand which class pairs cause the most errors for each architecture.


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
fig_cms, axes_cms = plt.subplots(1, 2, figsize=(16, 7))

for ax, name, hist in [
    (axes_cms[0], "LeNet-5", lenet_hist),
    (axes_cms[1], "AlexNet-CIFAR", alexnet_hist),
]:
    m_v = hist["model"]
    all_preds: list[int] = []
    all_labels: list[int] = []
    m_v.eval()
    with torch.no_grad():
        for imgs, lbls in test_loader:
            logits = m_v(imgs.to(device))
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(lbls.tolist())

    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=CIFAR_CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
    ax.set_title(f"{name}  (Test Acc: {hist['test_acc']:.2%})",
                 fontsize=10, fontweight="bold")
    hist["all_preds"]  = all_preds
    hist["all_labels"] = all_labels

plt.suptitle("Confusion Matrices — CIFAR-10 Test Set", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Per-class accuracy comparison ─────────────────────────────────────────────
print("\nPer-class accuracy comparison:")
print(f"  {'Class':<12s}  {'LeNet-5':>9s}  {'AlexNet':>9s}  {'Delta':>7s}")
print("-" * 45)
for cls_idx in range(NUM_CLASSES):
    accs = {}
    for name, hist in [("lenet", lenet_hist), ("alexnet", alexnet_hist)]:
        mask    = [l == cls_idx for l in hist["all_labels"]]
        n_c     = sum(p == l for p, l, m in zip(hist["all_preds"],
                                                 hist["all_labels"], mask) if m)
        n_total = sum(mask)
        accs[name] = n_c / max(n_total, 1)
    delta = accs["alexnet"] - accs["lenet"]
    print(f"  {CIFAR_CLASSES[cls_idx]:<12s}  {accs['lenet']:>8.2%}  "
          f"{accs['alexnet']:>8.2%}  {delta:>+6.2%}")


### 4.1b Top-K Accuracy & Prediction Confidence

Beyond top-1 accuracy, we examine whether the ground-truth class appears in the model's top-3 predictions, and whether the model is *confident* when it's right vs wrong.


In [ ]:
# ── Top-K accuracy and prediction confidence ─────────────────────────────────
# Top-1 accuracy is the standard metric, but top-3 accuracy (is the true class
# in the model's 3 best guesses?) gives a softer view of model quality.
# We also look at the softmax confidence distribution to detect overconfidence.

def topk_accuracy(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    k: int = 3,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    '''Compute top-1 and top-K accuracy, and collect confidence distributions.

    Args:
        model: Evaluated CNN model.
        dataloader: Test DataLoader.
        device: Compute device.
        k: K for top-K accuracy.

    Returns:
        Tuple of (top1_acc, topk_acc, correct_confs, wrong_confs) where
        correct_confs and wrong_confs are arrays of max-softmax confidence scores
        for correctly and incorrectly classified samples respectively.
    '''
    model.eval()
    top1_correct  = 0
    topk_correct  = 0
    total         = 0
    correct_confs: list[float] = []
    wrong_confs:   list[float] = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits   = model(images)
            probs    = torch.softmax(logits, dim=1)

            # Top-1
            top1_preds = logits.argmax(1)
            top1_mask  = top1_preds.eq(labels)
            top1_correct += top1_mask.sum().item()

            # Top-K
            topk_preds = logits.topk(k, dim=1).indices      # (N, k)
            labels_exp = labels.unsqueeze(1).expand_as(topk_preds)
            topk_mask  = topk_preds.eq(labels_exp).any(dim=1)
            topk_correct += topk_mask.sum().item()

            # Confidence = max softmax probability
            max_probs = probs.max(dim=1).values.cpu().numpy()
            top1_mask_cpu = top1_mask.cpu().numpy()
            correct_confs.extend(max_probs[top1_mask_cpu].tolist())
            wrong_confs.extend(max_probs[~top1_mask_cpu].tolist())

            total += labels.size(0)

    return (
        top1_correct / total,
        topk_correct / total,
        np.array(correct_confs),
        np.array(wrong_confs),
    )


K = 3
results_topk = {}
for name, hist in [("LeNet-5", lenet_hist), ("AlexNet-CIFAR", alexnet_hist)]:
    top1, topk, c_conf, w_conf = topk_accuracy(
        hist["model"], test_loader, device, k=K
    )
    results_topk[name] = {
        "top1": top1, "topk": topk,
        "correct_confs": c_conf, "wrong_confs": w_conf,
    }
    print(f"{name:<18s}  Top-1: {top1:.2%}  Top-{K}: {topk:.2%}  "
          f"(correct conf mean: {c_conf.mean():.3f}  "
          f"wrong conf mean: {w_conf.mean():.3f})")

# ── Plot confidence distributions ─────────────────────────────────────────────
fig_conf, axes_conf = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, res) in zip(axes_conf, results_topk.items()):
    bins = np.linspace(0, 1, 30)
    ax.hist(res["correct_confs"], bins=bins, alpha=0.65, color="#2ca02c",
            label=f"Correct (n={len(res['correct_confs']):,})", density=True)
    ax.hist(res["wrong_confs"],   bins=bins, alpha=0.65, color="#d62728",
            label=f"Wrong   (n={len(res['wrong_confs']):,})", density=True)
    ax.axvline(res["correct_confs"].mean(), color="#2ca02c", ls="--", lw=1.5,
               label=f"Correct mean={res['correct_confs'].mean():.2f}")
    ax.axvline(res["wrong_confs"].mean(),   color="#d62728", ls="--", lw=1.5,
               label=f"Wrong mean={res['wrong_confs'].mean():.2f}")
    ax.set_xlabel("Max softmax confidence", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.set_title(f"{name}: Confidence Distribution", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Prediction Confidence: Correct vs Incorrect Samples",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Top-K summary table ────────────────────────────────────────────────────────
print(f"\nTop-K Accuracy Summary:")
print(f"  {'Model':<18s}  {'Top-1':>7s}  {'Top-3':>7s}  {'Top-3 Gain':>11s}")
print("-" * 52)
for name, res in results_topk.items():
    gain = res["topk"] - res["top1"]
    print(f"  {name:<18s}  {res['top1']:>7.2%}  {res['topk']:>7.2%}  {gain:>+10.2%}")
print("\nTop-3 gain > Top-1 means the model 'knows' the right class but isn't top-confident.")


### 4.1c Confusion Pairs & Misclassified Images

Which class pairs cause the most confusion for each model? Visualising misclassified samples reveals whether the errors are 'reasonable' (visually similar classes) or surprising.


In [ ]:
# ── Top-confusion class pairs and worst misclassified images ─────────────────
# For each model, find the top-5 most frequent confusion pairs and show
# sample images where both models disagree with the ground truth.

def get_top_confusions(
    all_labels: list[int],
    all_preds: list[int],
    class_names: list[str],
    top_n: int = 5,
) -> list[dict]:
    '''Return the top-N most confused (true, predicted) class pairs.

    Args:
        all_labels: Ground-truth class indices.
        all_preds: Predicted class indices.
        class_names: List of class name strings.
        top_n: Number of confusion pairs to return.

    Returns:
        List of dicts with keys: true_class, pred_class, count.
    '''
    pair_counts: dict[tuple[int, int], int] = {}
    for lbl, pred in zip(all_labels, all_preds):
        if lbl != pred:
            pair_counts[(lbl, pred)] = pair_counts.get((lbl, pred), 0) + 1
    sorted_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])[:top_n]
    return [
        {"true_class": class_names[p[0]], "pred_class": class_names[p[1]], "count": c}
        for p, c in sorted_pairs
    ]


print("Top-5 confusion pairs:")
print(f"\n{'Rank':>4s}  {'Model':<16s}  {'True':>12s}  ->  {'Predicted':>12s}  {'Count':>6s}")
print("-" * 58)
for name, hist in [("LeNet-5", lenet_hist), ("AlexNet-CIFAR", alexnet_hist)]:
    confusions = get_top_confusions(
        hist["all_labels"], hist["all_preds"], CIFAR_CLASSES
    )
    for rank, conf in enumerate(confusions, 1):
        print(f"  {rank:>2d}  {name:<16s}  {conf['true_class']:>12s}  ->  "
              f"{conf['pred_class']:>12s}  {conf['count']:>6d}")
    print()

# ── Show misclassified samples for a specific confusion pair ──────────────────
# Pick the worst confusion pair of LeNet-5 and show images
lenet_confusions  = get_top_confusions(
    lenet_hist["all_labels"], lenet_hist["all_preds"], CIFAR_CLASSES
)
worst_true = CIFAR_CLASSES.index(lenet_confusions[0]["true_class"])
worst_pred = CIFAR_CLASSES.index(lenet_confusions[0]["pred_class"])

# Find test images: true=worst_true, LeNet predicted=worst_pred
misclassified_indices = [
    i for i, (lbl, pred) in enumerate(
        zip(lenet_hist["all_labels"], lenet_hist["all_preds"])
    )
    if lbl == worst_true and pred == worst_pred
][:8]

if misclassified_indices:
    raw_test = torchvision.datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=False,
        transform=transforms.ToTensor()
    )
    fig_err, axes_err = plt.subplots(1, len(misclassified_indices), figsize=(14, 2.5))
    if len(misclassified_indices) == 1:
        axes_err = [axes_err]
    for ax, idx in zip(axes_err, misclassified_indices):
        img_np = raw_test[idx][0].permute(1, 2, 0).numpy()
        ax.imshow(img_np)
        ax.set_title(f"True: {CIFAR_CLASSES[worst_true]}\nPred: {CIFAR_CLASSES[worst_pred]}",
                     fontsize=7)
        ax.axis("off")
    plt.suptitle(
        f"LeNet-5 Worst Confusion: '{lenet_confusions[0]['true_class']}' "
        f"predicted as '{lenet_confusions[0]['pred_class']}'  "
        f"({lenet_confusions[0]['count']} errors)",
        fontsize=9, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified samples for this pair in test set.")


### 4.2 First-Layer Filter Visualisation

The first convolutional layer learns filters that operate directly on raw RGB pixels.
- **LeNet-5 C1**: 6 filters of shape $3 \times 5 \times 5$ — broader spatial support.
- **AlexNet Block 1**: 64 filters of shape $3 \times 3 \times 3$ — more filters, smaller kernels.

Trained filters often resemble Gabor-like edge/orientation detectors — a pattern first
noted in AlexNet's famous 2012 filter visualisations.


In [ ]:
# ── First-layer filter visualisation ─────────────────────────────────────────
def show_conv1_filters(
    model: nn.Module,
    model_name: str,
    max_filters: int = 32,
) -> None:
    '''Plot first conv layer weight kernels as RGB images.

    Args:
        model: Trained CNN model (LeNet-5 or AlexNetCIFAR).
        model_name: Display name for the plot title.
        max_filters: Maximum number of filters to display.
    '''
    # Find first Conv2d
    first_conv: nn.Conv2d | None = None
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            first_conv = layer
            break
    if first_conv is None:
        print(f"{model_name}: No Conv2d found.")
        return

    weights = first_conv.weight.detach().cpu()   # (C_out, C_in, kH, kW)
    n_show  = min(max_filters, weights.shape[0])
    n_cols  = min(n_show, 16)
    n_rows  = math.ceil(n_show / n_cols)

    fig_f, axes_f = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    axes_f_flat = axes_f.ravel() if n_rows > 1 else list(axes_f)

    for k in range(n_show):
        kern = weights[k]       # (C_in, kH, kW)
        # Normalise to [0,1] for display
        if kern.shape[0] == 3:
            # RGB kernel — permute to (kH, kW, 3) and normalise
            kern_np = kern.permute(1, 2, 0).numpy()
            v_min, v_max = kern_np.min(), kern_np.max()
            kern_vis = (kern_np - v_min) / max(v_max - v_min, 1e-8)
            axes_f_flat[k].imshow(kern_vis)
        else:
            kern_np = kern[0].numpy()
            v = np.abs(kern_np).max() or 1.0
            axes_f_flat[k].imshow(kern_np, cmap="RdBu_r", vmin=-v, vmax=v)
        axes_f_flat[k].set_title(f"F{k}", fontsize=6)
        axes_f_flat[k].axis("off")

    for k in range(n_show, len(axes_f_flat)):
        axes_f_flat[k].axis("off")

    fig_f.suptitle(
        f"{model_name} — First Conv Layer Filters  "
        f"({n_show} of {weights.shape[0]}, shape {tuple(weights.shape[1:])})",
        fontsize=9, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # Filter statistics
    w_np = weights.numpy()
    print(f"{model_name} first conv filters: shape={weights.shape}  "
          f"mean={w_np.mean():.4f}  std={w_np.std():.4f}  "
          f"|w|_max={np.abs(w_np).max():.4f}")


show_conv1_filters(lenet_hist["model"],   "LeNet-5",       max_filters=6)
show_conv1_filters(alexnet_hist["model"], "AlexNet-CIFAR", max_filters=64)

# ── Intermediate feature maps on a sample image ───────────────────────────────
sample_img, sample_lbl = test_set[0]
sample_tensor = sample_img.unsqueeze(0).to(device)
sample_name   = CIFAR_CLASSES[sample_lbl]

fig_fms, axes_fms = plt.subplots(2, 8, figsize=(16, 5))
fig_fms.suptitle(
    f"First Conv Feature Maps — Sample: '{sample_name}'\n"
    "Row 0: LeNet-5 C1 (6 filters)   Row 1: AlexNet Block1 (first 8 of 64 filters)",
    fontsize=9, fontweight="bold",
)
for model_row, (model, n_show_maps, n_cols_here) in enumerate(
    [(lenet_hist["model"], 6, 6), (alexnet_hist["model"], 8, 8)]
):
    first_conv = next(m for m in model.modules() if isinstance(m, nn.Conv2d))
    with torch.no_grad():
        feat_maps = first_conv(sample_tensor)   # (1, C_out, H, W)
    for col_k in range(n_show_maps):
        feat = feat_maps[0, col_k].cpu().numpy()
        v    = np.abs(feat).max() or 1.0
        axes_fms[model_row][col_k].imshow(feat, cmap="RdBu_r", vmin=-v, vmax=v)
        axes_fms[model_row][col_k].set_title(f"ch{col_k}", fontsize=7)
        axes_fms[model_row][col_k].axis("off")
    for col_k in range(n_show_maps, 8):
        axes_fms[model_row][col_k].axis("off")

plt.tight_layout()
plt.show()


### 4.3 Final Summary & Design Insights

The accuracy gap between LeNet-5 and AlexNet on CIFAR-10 reflects 14 years of
architecture evolution.  The table below quantifies the key dimensions of that gap.


In [ ]:
# ── Final comparison table ────────────────────────────────────────────────────
lenet_p  = sum(p.numel() for p in lenet_hist["model"].parameters())
alex_p   = sum(p.numel() for p in alexnet_hist["model"].parameters())

final_rows = [
    {
        "Metric":        "Parameters",
        "LeNet-5":       f"{lenet_p:,}",
        "AlexNet-CIFAR": f"{alex_p:,}",
        "Note":          f"AlexNet has {alex_p // lenet_p}x more",
    },
    {
        "Metric":        "FLOPs / image",
        "LeNet-5":       f"{flops_lenet:,}",
        "AlexNet-CIFAR": f"{flops_alexnet:,}",
        "Note":          f"{flops_alexnet // max(flops_lenet,1)}x more",
    },
    {
        "Metric":        "Test Accuracy",
        "LeNet-5":       f"{lenet_hist['test_acc']:.4f}",
        "AlexNet-CIFAR": f"{alexnet_hist['test_acc']:.4f}",
        "Note":          f"+{alexnet_hist['test_acc'] - lenet_hist['test_acc']:.4f}",
    },
    {
        "Metric":        "Val Acc (epoch 10)",
        "LeNet-5":       f"{lenet_hist['val_accs'][-1]:.4f}",
        "AlexNet-CIFAR": f"{alexnet_hist['val_accs'][-1]:.4f}",
        "Note":          "",
    },
    {
        "Metric":        "Train Time (s)",
        "LeNet-5":       f"{lenet_hist['total_time']:.1f}",
        "AlexNet-CIFAR": f"{alexnet_hist['total_time']:.1f}",
        "Note":          f"{alexnet_hist['total_time']/max(lenet_hist['total_time'],1):.1f}x slower",
    },
    {
        "Metric":        "Activation",
        "LeNet-5":       "Tanh",
        "AlexNet-CIFAR": "ReLU",
        "Note":          "ReLU avoids Tanh saturation",
    },
    {
        "Metric":        "Regularisation",
        "LeNet-5":       "None",
        "AlexNet-CIFAR": "Dropout (p=0.5)",
        "Note":          "Dropout crucial at this scale",
    },
]

final_df = pd.DataFrame(final_rows)
print("Final comparison — LeNet-5 vs AlexNet-CIFAR:")
print(final_df.to_string(index=False))

# Bar chart: test accuracy side by side
fig_bar, ax_bar = plt.subplots(figsize=(6, 4))
names_bar = ["LeNet-5", "AlexNet-CIFAR"]
accs_bar  = [lenet_hist["test_acc"] * 100, alexnet_hist["test_acc"] * 100]
bar_cols  = ["#1f77b4", "#d62728"]
bars      = ax_bar.bar(names_bar, accs_bar, color=bar_cols, edgecolor="k", lw=0.8)
for bar, acc in zip(bars, accs_bar):
    ax_bar.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3, f"{acc:.2f}%",
                ha="center", va="bottom", fontsize=10, fontweight="bold")
ax_bar.set_ylabel("Test Accuracy (%)", fontsize=11)
ax_bar.set_title("CIFAR-10 Test Accuracy: LeNet-5 vs AlexNet-CIFAR",
                 fontsize=11, fontweight="bold")
ax_bar.set_ylim(0, 105)
ax_bar.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Design insights ────────────────────────────────────────────────────────────
lenet_delta  = lenet_hist["val_accs"][-1] - lenet_hist["val_accs"][0]
alexnet_delta = alexnet_hist["val_accs"][-1] - alexnet_hist["val_accs"][0]
print(f"\nVal accuracy improvement over 10 epochs:")
print(f"  LeNet-5       : epoch1 {lenet_hist['val_accs'][0]:.2%} "
      f"-> epoch10 {lenet_hist['val_accs'][-1]:.2%}  (delta {lenet_delta:+.2%})")
print(f"  AlexNet-CIFAR : epoch1 {alexnet_hist['val_accs'][0]:.2%} "
      f"-> epoch10 {alexnet_hist['val_accs'][-1]:.2%}  (delta {alexnet_delta:+.2%})")
print("\nConclusion: AlexNet's higher capacity + ReLU + Dropout yields better accuracy,")
print("but takes longer to train. Both would benefit from more epochs and a LR schedule.")


## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **LeNet-5 (1998):** Tanh activations, AvgPool subsampling, ~60 K parameters.
  Proved that learnable convolutional filters outperform hand-crafted features.
  Still achieves reasonable accuracy on CIFAR-10 (~55–65 %) despite being designed
  for 28×28 single-channel digits.
- **AlexNet (2012):** ReLU eliminates Tanh saturation; MaxPool provides stronger
  translation invariance; Dropout prevents overfitting in large FC layers.
  Achieves ~75–85 % on CIFAR-10 with 10× more parameters.
- **ReLU vs Tanh:** ReLU allows gradients to flow without saturation for any positive
  input, enabling deeper and wider networks.  This single change contributed
  significantly to AlexNet's faster training.
- **Data augmentation:** Random crops and horizontal flips effectively double/triple
  the training set diversity — a regularisation technique that costs nothing at inference.
- **Scale matters, but so does architecture:** AlexNet's accuracy advantage over
  LeNet-5 comes from a combination of deeper architecture, better activations, and
  explicit regularisation — not just raw parameter count.

### What's Next

→ **06-04 (VGGNet & Deep CNN Design)** pushes depth further: replacing large kernels
with stacks of 3×3 convolutions and exploring how many layers are useful before
vanishing gradients become a problem (addressed by ResNet in 06-05).
